importing libraries and the dataset

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = "iframe"
import plotly.express as px
from textblob import TextBlob
import plotly.graph_objects as go

In [ ]:
df=pd.read_csv('/Users/abhimanyuchettiar/Downloads/zomato.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
#all plotly expressions look like this
#px.chart_type(data_frame=df, x="col1", y="col2")


In [ ]:
df.columns

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
df = pd.read_csv('/Users/abhimanyuchettiar/Downloads/zomato.csv')
df.drop_duplicates(inplace=True)
def clean_rate(x):
    if(x == 'NEW' or x == '-'):
        return np.nan
    else:
        x = str(x).split('/')[0]
        return float(x)

df['rate'] = df['rate'].apply(clean_rate)
df['rate'] = df['rate'].fillna(df['rate'].mean())

# 3. Clean 'Cost' column (Remove commas from "1,200")
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(str).apply(lambda x: x.replace(',', ''))
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(float)

In [ ]:
# Grouping by location and summing the votes
location_votes = df.groupby('location')['votes'].sum().reset_index().sort_values(by='votes', ascending=False)

# Taking the Top 20 locations to keep the chart clean
fig = px.bar(location_votes.head(20), 
             x='location', 
             y='votes', 
             title='Total Votes per Location (Customer Engagement)',
             color='votes', # Adds a color gradient based on value
             color_continuous_scale='Oranges',
             template='plotly_dark')

fig.show()

**Analysis Note**: "We observe that Koramangala and BTM lead in total votes. This indicates these areas are not just restaurant hubs, but have a highly engaged customer base that frequently interacts with the Zomato platform."

In [ ]:
distinct_count = df['name'].nunique()
distinct_count

In [ ]:
print(df.index)

In [ ]:
top_locations = df['location'].value_counts().nlargest(10).reset_index()
top_locations.columns = ['Location', 'Count']

fig = px.bar(top_locations, x='Location', y='Count', 
             title='Top 10 Locations with Highest Number of Restaurants',
             text='Count', color='Count', color_continuous_scale='Reds')
fig.update_traces(textposition='outside')
fig.show()

This shows market saturation of the restaurant business by region

In [ ]:
fig = px.scatter(df, x='approx_cost(for two people)', y='rate', 
                 color='online_order', opacity=0.5,
                 title='Cost for Two vs. Rating',
                 labels={'approx_cost(for two people)': 'Approx Cost (Two People)', 'rate': 'Rating'},
                 trendline="ols") # Note: requires 'statsmodels' installed
fig.show()

this checks if paying more leads to a better restaurant experience, also shows almost no online orders when the cost is very high

In [ ]:
fig = px.box(df, x='online_order', y='rate', color='online_order',
             title='Impact of Online Ordering on Ratings',
             labels={'online_order': 'Offers Online Order', 'rate': 'Rating'},
             template='plotly_dark', color_discrete_sequence=['#FF3333', '#333333'])
fig.show()

This shows the distribution of ratings based on whether a restaurant takes online orders. It helps identify if digital accessibility correlates with customer satisfaction.

In [ ]:
fig = px.sunburst(df, path=['listed_in(type)', 'online_order'], 
                  values='votes', color='listed_in(type)',
                  title='Hierarchy of Restaurant Types and Online Order Volume')
fig.show()

This tells us which category of food (e.g., Desserts, Drinks & Nightlife) generates the most engagement (votes).

In [ ]:
Implementing machine learning models

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn import metrics

# 1. Encoding categorical variables
def encode_df(df):
    for column in df.columns[~df.columns.isin(['rate', 'approx_cost(for two people)', 'votes'])]:
        df[column] = df[column].factorize()[0]
    return df

df_en = encode_df(df.copy())

# 2. Defining X (Features) and y (Target)
X = df_en.drop(['rate', 'name', 'reviews_list'], axis=1) # Dropping non-predictive text
y = df_en['rate']

# 3. Splitting into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
df_en = encode_df(df.copy()) 
df_en.dropna(inplace=True)
X = df_en.drop(['rate', 'name', 'reviews_list'], axis=1)
y = df_en['rate']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(),
    "Random Forest": RandomForestRegressor(n_estimators=100)
}

results = {}
fig = go.Figure()

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae = metrics.mean_absolute_error(y_test, preds)
    results[name] = mae
    fig.add_trace(go.Scatter(x=y_test, y=preds, mode='markers', name=f'{name} (MAE: {mae:.2f})', opacity=0.5))
fig.show()

In [ ]:
# Visualize the error scores
error_df = pd.DataFrame(list(results.items()), columns=['Model', 'MAE'])

fig_error = px.bar(error_df, x='Model', y='MAE', color='MAE',
                   title='Model Performance (Lower is Better)',
                   color_continuous_scale='Reds', template='plotly_dark')
fig_error.show()